# F-OAL — CIFAR-100 Multi-Seed Table 4 Ablation Study (Seeds 42, 123, 2026)

- **Paper:** *F-OAL: Forward-only Online Analytic Learning with Fast Training and Low Memory Footprint in Class Incremental Learning* (NeurIPS 2024, Table 4)
- **Group Repository:** `https://github.com/caoban123/F-OAL`
- **Branch:** `experiment/cifar100-multiseed-ablation`
- **Protocol (Fixed C3-D / Table 4 Verified):**
  - Backbone: `vit_base_patch16_224.augreg2_in21k_ft_in1k`
  - Preprocessing: Bilinear resize (224x224), Raw tensor `[0, 1]` (`normalize: false`)
  - Smooth Projection: `official_uniform`, `dim=1000` (when enabled)
  - Online RLS: `gamma=1.0`, `float64`, `batch_size=10`, 1-pass, zero replay
- **Compute Plan:**
  - **Seed 42 (4 runs):** Reused from verified artifacts (`3v53tmq5`, `shl4p10l`, `rkedlkkk`, `k01pp8st`). Never re-run or overwritten.
  - **Seeds 123 & 2026 (8 new Full GPU runs):** `full`, `no_ff`, `no_sp`, `no_ff_no_sp` for each seed.

## 1. Preflight Checks (GPU, Git, Dependencies, Dataset, Configs, Cache)

In [ ]:
import json, os, platform, shutil, subprocess, sys
from pathlib import Path
import torch

print(f'Python   : {sys.version}')
print(f'Platform : {platform.platform()}')
print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
if not torch.cuda.is_available():
    raise RuntimeError('FATAL: CUDA GPU is required. Switch Colab runtime to GPU (T4/L4/A100).')

print(f'GPU Name : {torch.cuda.get_device_name(0)}')
print(f'VRAM (GB): {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f}')
free_gb = shutil.disk_usage('.').free / (1024**3)
print(f'Free Disk: {free_gb:.2f} GB')
if free_gb < 5.0:
    raise RuntimeError(f'Insufficient disk space ({free_gb:.2f} GB < 5 GB).')

In [ ]:
REPO_DIR = Path('/content/F-OAL')
BRANCH_NAME = 'experiment/cifar100-multiseed-ablation'

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '-b', BRANCH_NAME, 'https://github.com/caoban123/F-OAL.git', str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', 'fetch', 'origin'], cwd=str(REPO_DIR), check=True)
    subprocess.run(['git', 'checkout', BRANCH_NAME], cwd=str(REPO_DIR), check=True)
    subprocess.run(['git', 'pull', 'origin', BRANCH_NAME], cwd=str(REPO_DIR), check=True)

os.chdir(str(REPO_DIR))
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

GIT_SHA = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print(f'Repository ready at {REPO_DIR} | Commit: {GIT_SHA}')

In [ ]:
!pip install -q timm pyyaml pytest datasets torchvision matplotlib
!pip install -q --upgrade wandb

import wandb
WANDB_BASE_URL = 'https://api.forge.coreweave.com'
os.environ['WANDB_BASE_URL'] = WANDB_BASE_URL
try:
    wandb.login(host=WANDB_BASE_URL)
    USE_WANDB = True
except Exception as exc:
    print(f'W&B login skipped/failed ({exc}); using offline mode.')
    os.environ['WANDB_MODE'] = 'offline'
    USE_WANDB = True

In [ ]:
# Verify CIFAR-100 dataset loading, Seed 42 audit, and multi-seed runner status
from src.foal.datasets import load_cifar100
train_ds, test_ds = load_cifar100()
print(f'CIFAR-100 loaded: train={len(train_ds)}, test={len(test_ds)}')

subprocess.run([sys.executable, '-m', 'scripts.run_cifar100_multiseed_ablation', '--status-only'], check=True)
subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/test_multiseed_ablation.py'], check=True)

## 2. Pilot Smoke Test (Isolated Directory & W&B Group)

In [ ]:
# Runs 2 tasks x 50 samples for seed=123 (full & no_ff) into results/cifar100_multiseed_pilot
subprocess.run([
    sys.executable, '-m', 'scripts.run_cifar100_multiseed_ablation',
    '--pilot', '--seeds', '123', '--ablations', 'full', 'no_ff',
    '--device', 'cuda', '--wandb',
], check=True)

## 3. Full Experiments (8 Runs: Seeds 123 & 2026 x 4 Ablations)

In [ ]:
# Resumable: already-completed & validated runs are automatically skipped.
# Feature cache isolates by (backbone, interp, norm, ff/noff, class_order_seed, stream_seed, task_id).
subprocess.run([
    sys.executable, '-m', 'scripts.run_cifar100_multiseed_ablation',
    '--seeds', '123', '2026', '--device', 'cuda', '--wandb',
], check=True)

## 4. Independent Validation & Statistical Analysis

In [ ]:
subprocess.run([
    sys.executable, '-m', 'scripts.analyze_cifar100_multiseed',
    '--input-root', 'results/cifar100_multiseed',
], check=True)

summary = json.loads(Path('results/cifar100_multiseed/analysis/multiseed_summary.json').read_text(encoding='utf-8'))
print('\n=== MULTI-SEED SUMMARY (Average Incremental Accuracy %) ===')
for k, row in summary['summary_tables']['average_incremental_accuracy_pct']['rows'].items():
    p = row['per_seed']
    print(f"{row['label']:<18} | s42={p['42']:.4f}% | s123={p['123']:.4f}% | s2026={p['2026']:.4f}% | {row['mean_std_str']}")

print('\n=== PAIRED COMPONENT CONTRIBUTIONS (pp) ===')
pe = summary['paired_effects']['average_incremental_accuracy_pct']
for c_key in ('delta_ff', 'delta_sp', 'delta_both'):
    c = pe[c_key]
    print(f"{c_key:<12}: {c['mean_std_str']} (positive in {c['positive_seeds']}/3 seeds)")

In [ ]:
# Display generated research figures inline
from IPython.display import Image, display
for fig_name in [
    'fig1_multiseed_ablation_accuracy.png',
    'fig2_paired_component_contributions.png',
    'fig3_multiseed_accuracy_progression.png',
    'fig4_accuracy_vs_forgetting.png',
]:
    p = Path('results/cifar100_multiseed/analysis') / fig_name
    if p.exists():
        print(fig_name)
        display(Image(filename=str(p)))

## 5. Export Verification Archive (`foal_cifar100_multiseed_ablation_results.zip`)

In [ ]:
import hashlib, zipfile
from src.foal.environment import get_environment_info

export_meta_dir = Path('results/cifar100_multiseed/export_metadata')
export_meta_dir.mkdir(parents=True, exist_ok=True)
(export_meta_dir / 'environment_snapshot.json').write_text(json.dumps(get_environment_info(), indent=2), encoding='utf-8')

zip_path = Path('foal_cifar100_multiseed_ablation_results.zip')
include_roots = [
    Path('results/cifar100_multiseed'),
    Path('configs/table4_a0_full_foal_cifar100.yaml'),
    Path('configs/table4_a1_without_ff_cifar100.yaml'),
    Path('configs/table4_a2_without_sp_cifar100.yaml'),
    Path('configs/table4_a3_without_both_cifar100.yaml'),
]

checksums = {}
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for root in include_roots:
        files = [root] if root.is_file() else sorted(p for p in root.rglob('*') if p.is_file() and not p.suffix == '.pt')
        for fp in files:
            rel = fp.as_posix()
            h = hashlib.sha256(fp.read_bytes()).hexdigest()
            checksums[rel] = h
            zf.write(fp, arcname=rel)
    manifest_bytes = json.dumps({'git_commit': GIT_SHA, 'files': checksums}, indent=2).encode('utf-8')
    zf.writestr('results/cifar100_multiseed/archive_sha256_manifest.json', manifest_bytes)

print(f'Exported archive: {zip_path} ({zip_path.stat().st_size:,} bytes, {len(checksums)} files)')
kaggle_working = Path('/kaggle/working')
if kaggle_working.exists():
    import shutil as _shutil
    _kw_zip = kaggle_working / zip_path.name
    _shutil.copy2(zip_path, _kw_zip)
    print(f'Copied archive to Kaggle Output: {_kw_zip}')
try:
    from google.colab import files
    files.download(str(zip_path))
except ImportError:
    print('Not running inside Google Colab; archive saved locally.')